# Cell 1 — Check GPU

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No GPU found. In Colab: Runtime > Change runtime type > GPU")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
bf16 supported: True


# Cell 2 — Install dependencies

In [2]:
!pip -q install -U datasets transformers sentencepiece tqdm accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.6 MB/s eta 0:00:00


# Cell 3 — Imports

In [3]:
import os
import math
import numpy as np
from tqdm.auto import tqdm
from dataclasses import dataclass
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer

# Cell 4 — Local Colab folders, no Drive

In [4]:
PROJECT_DIR = "/content/tinystories_mistral_slm"
DATA_DIR = f"{PROJECT_DIR}/data"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
TOKENIZER_DIR = f"{PROJECT_DIR}/tokenizer"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Checkpoint directory:", CKPT_DIR)

Project directory: /content/tinystories_mistral_slm
Data directory: /content/tinystories_mistral_slm/data
Checkpoint directory: /content/tinystories_mistral_slm/checkpoints


# Cell 5 — Load TinyStories dataset

In [5]:
ds = load_dataset("roneneldan/TinyStories")

print(ds)
print(ds["train"][0]["text"][:500])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


# Cell 6 — Load Mistral tokenizer instead of GPT-2 tokenizer

In [6]:
TOKENIZER_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print("Tokenizer:", TOKENIZER_NAME)
print("Vocab size:", tokenizer.vocab_size)
print("BOS token:", tokenizer.bos_token, tokenizer.bos_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer: mistralai/Mistral-7B-v0.1
Vocab size: 32000
BOS token: <s> 1
EOS token: </s> 2
PAD token: None None


# Cell 7 — Test Mistral tokenizer

In [7]:
sample_text = "Once upon a time there was a pumpkin."

ids = tokenizer.encode(sample_text, add_special_tokens=False)

print("Text:", sample_text)
print("Token IDs:", ids)
print("Number of tokens:", len(ids))
print("Decoded:", tokenizer.decode(ids))

Text: Once upon a time there was a pumpkin.
Token IDs: [5713, 3714, 264, 727, 736, 403, 264, 12048, 7510, 28723]
Number of tokens: 10
Decoded: Once upon a time there was a pumpkin.


# Cell 8 — Save tokenizer locally

In [8]:
tokenizer.save_pretrained(TOKENIZER_DIR)

print("Tokenizer saved locally at:", TOKENIZER_DIR)

Tokenizer saved locally at: /content/tinystories_mistral_slm/tokenizer


# Cell 9 — Tokenize TinyStories and create train.bin / val.bin

In [9]:
train_bin_path = f"{DATA_DIR}/train.bin"
val_bin_path = f"{DATA_DIR}/val.bin"

# Important:
# Mistral vocab_size is 32000, so uint16 is safe.
# uint16 supports values up to 65535.

eos_id = tokenizer.eos_token_id

def process(example):
    ids = tokenizer.encode(
        example["text"],
        add_special_tokens=False
    )

    # Add EOS between stories so the model learns story boundaries.
    ids.append(eos_id)

    return {
        "ids": ids,
        "len": len(ids)
    }


split_to_fname = {
    "train": train_bin_path,
    "validation": val_bin_path
}

if not (os.path.exists(train_bin_path) and os.path.exists(val_bin_path)):

    tokenized = ds.map(
        process,
        remove_columns=["text"],
        desc="Tokenizing TinyStories with Mistral tokenizer",
        num_proc=2,
    )

    for split, dset in tokenized.items():
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = split_to_fname[split]

        print(f"Writing {filename}")
        print(f"Total tokens: {arr_len:,}")

        arr = np.memmap(
            filename,
            dtype=np.uint16,
            mode="w+",
            shape=(arr_len,)
        )

        total_batches = 1024
        idx = 0

        for batch_idx in tqdm(
            range(total_batches),
            desc=f"Writing {split}"
        ):
            batch = dset.shard(
                num_shards=total_batches,
                index=batch_idx,
                contiguous=True
            ).with_format("numpy")

            arr_batch = np.concatenate(batch["ids"])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        arr.flush()

else:
    print("train.bin and val.bin already exist. Skipping tokenization.")

Tokenizing TinyStories with Mistral tokenizer (num_proc=2):   0%|          | 0/2119719 [00:00<?, ? examples/s]

Tokenizing TinyStories with Mistral tokenizer (num_proc=2):   0%|          | 0/21990 [00:00<?, ? examples/s]

Writing /content/tinystories_mistral_slm/data/train.bin
Total tokens: 492,709,180


Writing train:   0%|          | 0/1024 [00:00<?, ?it/s]

Writing /content/tinystories_mistral_slm/data/val.bin
Total tokens: 4,955,713


Writing validation:   0%|          | 0/1024 [00:00<?, ?it/s]

# Cell 10 — Check tokenized files

In [10]:
train_data_check = np.memmap(train_bin_path, dtype=np.uint16, mode="r")
val_data_check = np.memmap(val_bin_path, dtype=np.uint16, mode="r")

print("Train tokens:", len(train_data_check))
print("Val tokens:", len(val_data_check))
print("First 20 train tokens:", train_data_check[:20].tolist())

decoded_sample = tokenizer.decode(train_data_check[:100].astype(np.int64).tolist())
print(decoded_sample)

Train tokens: 492709180
Val tokens: 4955713
First 20 train tokens: [2387, 1370, 28725, 264, 1628, 2746, 5160, 25492, 1419, 264, 25710, 297, 559, 2003, 28723, 985, 2580, 378, 403, 3796]
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix


# Cell 11 — Training settings

In [11]:
# Tokenizer / context settings
vocab_size = tokenizer.vocab_size
block_size = 2048

# SLM architecture
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.1
bias = True

# Training hyperparameters
learning_rate = 3e-4
max_iters = 5000
warmup_steps = 200
min_lr = 3e-5

eval_interval = 250
eval_iters = 50

# For block_size 2048, keep batch_size small in Colab
batch_size = 1
gradient_accumulation_steps = 16

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
device_type = "cuda" if device.startswith("cuda") else "cpu"

use_bf16 = device_type == "cuda" and torch.cuda.is_bf16_supported()
dtype = "bfloat16" if use_bf16 else "float16"

ptdtype = torch.bfloat16 if dtype == "bfloat16" else torch.float16

ctx = (
    nullcontext()
    if device_type == "cpu"
    else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device_type == "cuda" and dtype == "float16")
)

torch.manual_seed(42)

print("Device:", device)
print("dtype:", dtype)
print("vocab_size:", vocab_size)
print("block_size:", block_size)
print("batch_size:", batch_size)
print("gradient_accumulation_steps:", gradient_accumulation_steps)

Device: cuda
dtype: bfloat16
vocab_size: 32000
block_size: 2048
batch_size: 1
gradient_accumulation_steps: 16


# Cell 12 — Load binary data

In [12]:
train_data = np.memmap(train_bin_path, dtype=np.uint16, mode="r")
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode="r")

print("Train tokens:", len(train_data))
print("Val tokens:", len(val_data))

Train tokens: 492709180
Val tokens: 4955713


# Cell 13 — Batch function

In [13]:
def get_batch(split: str):
    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        torch.from_numpy(
            data[i : i + block_size].astype(np.int64)
        )
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(
            data[i + 1 : i + 1 + block_size].astype(np.int64)
        )
        for i in ix
    ])

    if device_type == "cuda":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x = x.to(device)
        y = y.to(device)

    return x, y


X, Y = get_batch("train")

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Max token id in batch:", X.max().item())
print("Vocab size:", vocab_size)

X shape: torch.Size([1, 2048])
Y shape: torch.Size([1, 2048])
Max token id in batch: 28758
Vocab size: 32000


# Cell 14 — Model architecture

In [14]:
class LayerNorm(nn.Module):
    def __init__(self, ndim: int, bias: bool):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.layer_norm(
            x,
            self.weight.shape,
            self.weight,
            self.bias,
            1e-5
        )


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(
            config.n_embd,
            3 * config.n_embd,
            bias=config.bias
        )

        self.c_proj = nn.Linear(
            config.n_embd,
            config.n_embd,
            bias=config.bias
        )

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.n_head = config.n_head
        self.n_embd = config.n_embd

        self.flash = hasattr(F, "scaled_dot_product_attention")
        self.bias = None

        if not self.flash:
            self.register_buffer(
                "bias",
                torch.tril(
                    torch.ones(config.block_size, config.block_size)
                ).view(1, 1, config.block_size, config.block_size)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
            )
        else:
            assert self.bias is not None

            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))

            att = att.masked_fill(
                self.bias[:, :, :T, :T] == 0,
                float("-inf")
            )

            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)

            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))

        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.c_fc = nn.Linear(
            config.n_embd,
            4 * config.n_embd,
            bias=config.bias
        )

        self.gelu = nn.GELU()

        self.c_proj = nn.Linear(
            4 * config.n_embd,
            config.n_embd,
            bias=config.bias
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)

        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))

        return x


@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True


class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()

        self.config = config

        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                drop=nn.Dropout(config.dropout),
                h=nn.ModuleList([
                    Block(config) for _ in range(config.n_layer)
                ]),
                ln_f=LayerNorm(config.n_embd, config.bias),
            )
        )

        self.lm_head = nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False
        )

        # Weight tying
        self.transformer["wte"].weight = self.lm_head.weight

        self.apply(self._init_weights)

        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                nn.init.normal_(
                    p,
                    mean=0.0,
                    std=0.02 / math.sqrt(2 * config.n_layer)
                )

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor | None = None
    ):
        device = idx.device
        b, t = idx.size()

        assert t <= self.config.block_size

        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer["wte"](idx)
        pos_emb = self.transformer["wpe"](pos)

        x = self.transformer["drop"](tok_emb + pos_emb)

        for block in self.transformer["h"]:
            x = block(x)

        x = self.transformer["ln_f"](x)

        if targets is not None:
            logits = self.lm_head(x)

            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

            return logits, loss

        logits = self.lm_head(x[:, [-1], :])

        return logits, None

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None
    ):
        for _ in range(max_new_tokens):
            idx_cond = (
                idx
                if idx.size(1) <= self.config.block_size
                else idx[:, -self.config.block_size:]
            )

            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k is not None:
                v, _ = torch.topk(
                    logits,
                    min(top_k, logits.size(-1))
                )

                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Cell 15 — Create the model

In [15]:
config = GPTConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    dropout=dropout,
    bias=bias
)

model = GPT(config).to(device)

total_params = sum(p.numel() for p in model.parameters())

print("Model created.")
print("Model parameters:", f"{total_params / 1e6:.2f}M")
print("Vocab size:", config.vocab_size)
print("Context length:", config.block_size)

Model created.
Model parameters: 23.72M
Vocab size: 32000
Context length: 2048


# Cell 16 — Loss estimation function

In [16]:
@torch.inference_mode()
def estimate_loss(model):
    out = {}

    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)

            with ctx:
                _, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean().item()

    model.train()

    return out

# Cell 17 — Optimizer and scheduler

In [17]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1,
    eps=1e-9
)

scheduler_warmup = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)

scheduler_decay = CosineAnnealingLR(
    optimizer,
    T_max=max_iters - warmup_steps,
    eta_min=min_lr
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[scheduler_warmup, scheduler_decay],
    milestones=[warmup_steps]
)

print("Optimizer and scheduler ready.")

Optimizer and scheduler ready.


# Cell 18 — Training loop

In [18]:
best_val_loss = float("inf")
best_model_params_path = f"{CKPT_DIR}/best_model_params.pt"

train_loss_list = []
validation_loss_list = []

patience_evals = 10
min_delta = 0.0001
no_improve_count = 0

model.train()

optimizer.zero_grad(set_to_none=True)

for step in tqdm(range(max_iters), desc="Training SLM"):

    if step % eval_interval == 0 and step != 0:
        losses = estimate_loss(model)

        print(
            f"\nStep {step}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

        print(f"LR: {optimizer.param_groups[0]['lr']:.8f}")

        train_loss_list.append(losses["train"])
        validation_loss_list.append(losses["val"])

        if losses["val"] < best_val_loss - min_delta:
            best_val_loss = losses["val"]
            no_improve_count = 0

            torch.save(model.state_dict(), best_model_params_path)

            print("Saved new best model.")

        else:
            no_improve_count += 1

            print(
                f"No improvement count: "
                f"{no_improve_count}/{patience_evals}"
            )

            if no_improve_count >= patience_evals:
                print("Early stopping triggered.")
                break

    X, Y = get_batch("train")

    with ctx:
        logits, loss = model(X, Y)
        loss = loss / gradient_accumulation_steps

    if scaler.is_enabled():
        scaler.scale(loss).backward()
    else:
        loss.backward()

    if ((step + 1) % gradient_accumulation_steps == 0) or (step + 1 == max_iters):

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=0.5
        )

        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)

    scheduler.step()

print("Training finished.")
print("Best val loss:", best_val_loss)
print("Best checkpoint:", best_model_params_path)

Training SLM:   0%|          | 0/5000 [00:00<?, ?it/s]


Step 250: train loss 7.9419, val loss 7.9498
LR: 0.00029993
Saved new best model.

Step 500: train loss 5.9740, val loss 5.9647
LR: 0.00029741
Saved new best model.

Step 750: train loss 5.1683, val loss 5.1240
LR: 0.00029135
Saved new best model.

Step 1000: train loss 4.7068, val loss 4.7182
LR: 0.00028191
Saved new best model.

Step 1250: train loss 4.3900, val loss 4.3907
LR: 0.00026936
Saved new best model.

Step 1500: train loss 4.2074, val loss 4.2587
LR: 0.00025401
Saved new best model.

Step 1750: train loss 4.1022, val loss 4.1440
LR: 0.00023629
Saved new best model.

Step 2000: train loss 4.0929, val loss 4.1097
LR: 0.00021666
Saved new best model.

Step 2250: train loss 4.0100, val loss 3.9891
LR: 0.00019566
Saved new best model.

Step 2500: train loss 3.8967, val loss 3.9489
LR: 0.00017383
Saved new best model.

Step 2750: train loss 3.8971, val loss 3.9283
LR: 0.00015177
Saved new best model.

Step 3000: train loss 3.8609, val loss 3.8935
LR: 0.00013006
Saved new best mo

# Cell 19 — Save full checkpoint

In [19]:
full_checkpoint_path = f"{CKPT_DIR}/tinystories_mistral_slm_full.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": config.vocab_size,
        "block_size": config.block_size,
        "n_layer": config.n_layer,
        "n_head": config.n_head,
        "n_embd": config.n_embd,
        "dropout": config.dropout,
        "bias": config.bias,
    },
    "tokenizer_name": TOKENIZER_NAME,
    "train_loss_list": train_loss_list,
    "validation_loss_list": validation_loss_list,
}

torch.save(checkpoint, full_checkpoint_path)

print("Full checkpoint saved at:", full_checkpoint_path)

Full checkpoint saved at: /content/tinystories_mistral_slm/checkpoints/tinystories_mistral_slm_full.pt


# Cell 20 — Test generation with Mistral tokenizer

In [20]:
best_model = GPT(config).to(device)

best_model.load_state_dict(
    torch.load(best_model_params_path, map_location=device)
)

best_model.eval()

prompt = "Once upon a time there was a pumpkin."

context_ids = tokenizer.encode(
    prompt,
    add_special_tokens=False
)

context = torch.tensor(
    context_ids,
    dtype=torch.long,
    device=device
).unsqueeze(0)

with torch.no_grad():
    y = best_model.generate(
        context,
        max_new_tokens=120,
        temperature=0.7,
        top_k=50
    )

output_text = tokenizer.decode(y.squeeze().tolist())

print(output_text)

Once upon a time there was a pumpkin. One day, she liked to play in a little girl named Lily was too, Timmy gave him to play with his mommy asked her friends that day, Timmy't know what it was very happy to play with the day, Lily. Timmy's mom were two best friend. Timmy's mommy. Timmy's time, he had a way back to play with her mom, Lily.</s> Once upon a time, but she went to take a time, to play with her mommy. He found the dog named Lily was a little girl named Lily loved


# Cell 21 — Generate multiple samples

In [21]:
prompts = [
    "Once upon a time there was a little girl.",
    "Tom found a small box under the bed.",
    "The dog wanted to play in the garden.",
    "Lily was scared of the dark, but"
]

best_model.eval()

for prompt in prompts:
    context_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False
    )

    context = torch.tensor(
        context_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():
        y = best_model.generate(
            context,
            max_new_tokens=120,
            temperature=0.7,
            top_k=50
        )

    print("=" * 80)
    print(tokenizer.decode(y.squeeze().tolist()))

Once upon a time there was a little girl. The rabbit was a little girl named Lily. Timmy, you feel better. One day, "I't want to play with a time, Lily!"









Lily felt sad and play outside and Timmy and said, there was so pretty. Timmy was a time, Timmy was a little girl named Timmy's mom was very happy to be a time, happy because he't want to the little girl named Lily't worry, Timmy smiled and her friends. Timmy's mommy's mommy
Tom found a small box under the bed.

The tree. She said, "I's okay, "Mom said, "Yes, I can's not like you!" I will not like you. I's a big boy."

They have to play outside and said, "I love you can be so much. I am a big box. I am sorry, you and smiled. I can stay here, you.
Tom was a little girl named Lily. You did not want to use their mom.

They looked at the door and Dad and said. I just want to the door.
The dog wanted to play in the garden. The little girl was so much fun.</s> Once upon a little day, there was a little boy named Timmy loved to play

# Cell 22 — Download checkpoint manually

In [ ]:
from google.colab import files

files.download(full_checkpoint_path)

In [ ]:
files.download(best_model_params_path)

# Cell 23 — Reload full checkpoint later after upload

In [ ]:
from google.colab import files

uploaded = files.upload()

uploaded_checkpoint_path = list(uploaded.keys())[0]

print("Uploaded:", uploaded_checkpoint_path)

# Cell 24 — Load uploaded checkpoint

In [ ]:
checkpoint = torch.load(uploaded_checkpoint_path, map_location=device)

saved_config = checkpoint["config"]

config = GPTConfig(
    vocab_size=saved_config["vocab_size"],
    block_size=saved_config["block_size"],
    n_layer=saved_config["n_layer"],
    n_head=saved_config["n_head"],
    n_embd=saved_config["n_embd"],
    dropout=saved_config["dropout"],
    bias=saved_config["bias"],
)

tokenizer = AutoTokenizer.from_pretrained(checkpoint["tokenizer_name"])

model = GPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Checkpoint loaded successfully.")
print("Tokenizer:", checkpoint["tokenizer_name"])
print("Vocab size:", config.vocab_size)
print("Context length:", config.block_size)